In [5]:
import os
import camelot
import pandas as pd
from IPython.display import display
from tqdm import tqdm

folder = os.getcwd()
excels = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]

In [31]:
dfs = []
for i, pth in enumerate(tqdm(excels)):
    df_i = pd.read_excel(pth)
    df_i = df_i.loc[:, ['Part Number', 'Models']]
    dfs.append(df_i)

df_all = pd.concat(dfs, ignore_index=True)

100%|██████████| 19/19 [00:02<00:00,  8.40it/s]


In [35]:
def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None

In [40]:
def merge_parts(df):
    df = df.copy()

    agg_map = {
        "Models": join_ordered
    }

    df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

    return df_agg

In [46]:
df_all.info()

df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

df_letters = df_all[part_numeric.isna()].copy()
df_numbers = df_all[part_numeric.notna()].copy()

if not df_numbers.empty:
    df_numbers["Part Number"] = (
        part_numeric[part_numeric.notna()]
        .astype(int)
        .astype(str)
        .str.zfill(8)
    )

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15698 entries, 0 to 15697
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Part Number  15698 non-null  object
 1   Models       15698 non-null  object
dtypes: object(2)
memory usage: 245.4+ KB


In [51]:
df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)

In [52]:
df_merged

,Part Number,Models
0,00002362B,JCPT0708DCS
1,00000020,JCPT1523RTB; JCPT1823RTB
2,00000023,JCPT1523RTB; JCPT1823RTB
3,00000045,BA16NE; BA18NE
4,00000046,BA16NE; BA18NE
...,...,...
7246,93305316,BA36RT; BA41RT; BA44RT
7247,93305785,BA36RT; BA41RT; BA44RT
7248,93305786,BA36RT; BA41RT; BA44RT
7249,93305787,BA36RT; BA41RT; BA44RT


In [ ]:
df_merged.to_excel(r"final_модели.xlsx", sheet_name="Parts", index=False)

: 